# Notebook 07 — Response Bias (Supplementary Table 9)

Tests whether metacognitive measures are affected by *response bias* — a tendency to favour one response category (e.g., always say S2).

**Dataset**: Locke et al. (2020), 10 subjects, 7 conditions that systematically vary response criterion from very liberal to very conservative.

**Method**: One-way repeated-measures ANOVA (F(6,54)) across 7 conditions for each measure. If a measure is insensitive to response bias, F should be near 1.0. Criterion and d' serve as sanity checks — they *should* change across conditions.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


In [ ]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [ ]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


In [ ]:
import matplotlib.pyplot as plt

## Load precomputed Locke data

In [ ]:
lo_rb = np.load(os.path.join(OUT, 'locke_mle.npz'))['rb']   # (10, 7, 20)
print("Locke response-bias data:", lo_rb.shape, "  (subjects, conditions, measures)")
# Show condition-level criterion (should vary strongly)
print("\nMean Criterion per condition:")
for ci in range(7):
    print(f"  Condition {ci+1}: {np.nanmean(lo_rb[:, ci, 19]):.3f}")


## Supplementary Table 9 — RM-ANOVA

In [ ]:
REPORTED_T9 = {"meta-d'":1.472,"AUC2":0.742,"Criterion":12.185,"Confidence":0.482}

print("Supplementary Table 9 — Response bias: Locke (n=10, 7 conditions)")
print("=" * 70)
print(f"  {'Measure':<20} {'F(6,54)':>9} {'p':>8} {'sig':>4} {'η²p':>7} {'F MATLAB':>10}")
print("  " + "-"*62)
for m, name in enumerate(MEASURE_NAMES):
    data = lo_rb[:, :, m]
    complete = data[~np.any(np.isnan(data), axis=1)]
    if complete.shape[0] < 2:
        print(f"  {name:<20} {'NaN':>9}"); continue
    F, df_b, df_e, p, eta2 = rm_anova_1way(complete)
    rep_f = REPORTED_T9.get(name, float('nan'))
    match = ("✓" if abs(F-rep_f)<abs(rep_f)*0.05+0.05 else "~") if not np.isnan(rep_f) else ""
    print(f"  {name:<20} {F:9.3f} {p:8.4f} {p_stars(p):>4} {eta2:7.3f} "
          f"{rep_f:10.3f}  {match}" if not np.isnan(rep_f) else
          f"  {name:<20} {F:9.3f} {p:8.4f} {p_stars(p):>4} {eta2:7.3f}")


## Correlation with response bias

The mean absolute criterion |C| measures how far a subject's response is from neutral. A good metacognitive measure should be uncorrelated with |C|.

In [ ]:
from scipy.stats import pearsonr
print("Average correlation of each measure with |criterion| across conditions:")
for m, name in enumerate(MEASURE_NAMES):
    crit = np.abs(lo_rb[:, :, 19])   # (10, 7) criterion
    meas = lo_rb[:, :, m]            # (10, 7)
    rs = []
    for ci in range(7):
        x, y = crit[:, ci], meas[:, ci]
        ok   = ~np.isnan(x) & ~np.isnan(y)
        if ok.sum() >= 3:
            r, _ = pearsonr(x[ok], y[ok])
            rs.append(r)
    avg_r = np.mean(rs) if rs else np.nan
    print(f"  {name:<20}: r = {avg_r:.3f}" if not np.isnan(avg_r) else f"  {name:<20}: NaN")
